# Final Frozen MBPP+ Holdout — Gemma 4 Recovery Study

**Goal:** Evaluate *unchanged* C0/C1/C2 adapters on the **70 task IDs frozen in the Day 5 partition manifest**. Do not select new tasks, tune prompts, or train on these outcomes. The model receives **original public MBPP tests only**; official EvalPlus scoring runs **after generation**.

Attach **Day 2, Day 3 and Day 5** saved Kaggle notebook outputs via **Add Input** (or the three ZIP backups as private Kaggle datasets). This notebook only uses one T4 GPU. It checkpoints each generated result immediately.

**Security:** generated Python runs in resource-limited subprocesses, not a hardened sandbox; run only in a disposable runtime without sensitive data or secrets. Do not include HF tokens as literal cells.

## A — Libraries (do not restart mid-run)

In [ ]:
%pip install -q -U "transformers>=5.18.0" "peft>=0.19.0" accelerate datasets bitsandbytes safetensors tqdm evalplus matplotlib

## B — Runtime and hardware

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
from pathlib import Path
import json, sys, torch
OUT=Path('/kaggle/working/final_holdout_outputs')
OUT.mkdir(parents=True,exist_ok=True)
INPUTS=Path('/kaggle/working/final_holdout_inputs')
INPUTS.mkdir(parents=True,exist_ok=True)
MAX_NEW_TOKENS=384
print('CUDA:',torch.cuda.is_available(), 'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'Select GPU T4 x2 in Kaggle Accelerator settings'


## C — Recreate the exact tested evaluation scripts from Day 5

In [ ]:
%%writefile /kaggle/working/test_runner.py
"""Small, resource-limited function-code runner for Kaggle Day 1.

SECURITY: subprocess + resource limits are NOT a strong sandbox. Model-generated
code can still access local files or network unless the external runtime forbids
it. Never run with secrets, private data or elevated access. For a public-facing
benchmark or adversarial model, run in a truly isolated container/service.
"""
from __future__ import annotations

import json
import os
import resource
import subprocess
import sys
import tempfile
from pathlib import Path

# Entire driver runs in a separate Python interpreter; no generated code is exec'd
# in the notebook kernel. Child stdout is reserved for a JSON response.
_CHILD = r'''
import ast, contextlib, io, json, sys, traceback
payload = json.load(sys.stdin)
source = payload["code"]
checks = payload["tests"]
setup = payload.get("setup", "") or ""

def trace(e):
    return "".join(traceback.format_exception_only(type(e), e)).strip()[:1800]

namespace = {"__name__": "__main__"}
result = {"status": "passed", "passed": 0, "total": len(checks), "feedback": "All visible tests passed", "failures": []}

try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        exec(compile(source, "<candidate>", "exec"), namespace)
        if setup:
            exec(compile(setup, "<setup>", "exec"), namespace)
except BaseException as e:
    result["status"] = "syntax_error" if isinstance(e, SyntaxError) else "runtime_error"
    result["feedback"] = f"Failed before tests: {trace(e)}"
else:
    for test in checks:
        try:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                # For simple equality assertions, extract actual/expected values
                # to provide useful execution feedback without reevaluating them.
                tree = ast.parse(test, mode="exec")
                if (len(tree.body) == 1 and isinstance(tree.body[0], ast.Assert) and
                    isinstance(tree.body[0].test, ast.Compare) and
                    len(tree.body[0].test.ops) == 1 and
                    isinstance(tree.body[0].test.ops[0], ast.Eq)):
                    cmp = tree.body[0].test
                    actual = eval(compile(ast.Expression(cmp.left), "<actual>", "eval"), namespace)
                    expected = eval(compile(ast.Expression(cmp.comparators[0]), "<expected>", "eval"), namespace)
                    if actual != expected:
                        raise AssertionError(f"expected {repr(expected)[:220]}, got {repr(actual)[:220]}")
                else:
                    exec(compile(tree, "<test>", "exec"), namespace)
            result["passed"] += 1
        except BaseException as e:
            kind = "wrong_output" if isinstance(e, AssertionError) else "runtime_error"
            result["status"] = kind if result["status"] == "passed" else result["status"]
            result["failures"].append({"test": test[:500], "kind": kind, "message": trace(e)})
            # feedback limited to first failing test to avoid lengthy prompts
            if len(result["failures"]) >= 2:
                break
    if result["failures"]:
        result["feedback"] = "\n".join(
            f"Test: {f['test']}\nError: {f['message']}" for f in result["failures"]
        )[:1600]
print(json.dumps(result, ensure_ascii=False))
'''


def _limits():
    # POSIX-only; Kaggle is Linux. Limits accidental infinite loops and output size.
    resource.setrlimit(resource.RLIMIT_CPU, (3, 4))
    resource.setrlimit(resource.RLIMIT_AS, (1024 * 1024**2, 1024 * 1024**2))
    resource.setrlimit(resource.RLIMIT_FSIZE, (1024 * 1024, 1024 * 1024))


def run_visible_tests(code: str, tests: list[str], setup: str = "", timeout_s: int = 6) -> dict:
    """Execute candidate and visible asserts with timeout.

    Avoid passing hidden benchmark tests here: they must not feed back to the model.
    """
    payload = json.dumps({"code": code, "tests": list(tests), "setup": setup})
    # Pass a small environment, with no authentication tokens inherited.
    env = {"PATH": os.environ.get("PATH", "/usr/bin:/bin"),
           "PYTHONIOENCODING": "utf-8", "PYTHONDONTWRITEBYTECODE": "1"}
    with tempfile.TemporaryDirectory(prefix="gemma_code_") as temp:
        try:
            cp = subprocess.run(
                [sys.executable, "-I", "-S", "-B", "-c", _CHILD],
                input=payload, text=True, stdout=subprocess.PIPE,
                stderr=subprocess.PIPE, cwd=temp,
                env=env, preexec_fn=_limits, timeout=timeout_s,
            )
        except subprocess.TimeoutExpired:
            return {"status": "timeout", "passed": 0, "total": len(tests),
                    "feedback": f"Execution timed out after {timeout_s} seconds", "failures": []}
        if cp.returncode != 0:
            return {"status": "runtime_error", "passed": 0, "total": len(tests),
                    "feedback": f"Runner failed: {cp.stderr[-700:]} (exit={cp.returncode})", "failures": []}
        try:
            output = json.loads(cp.stdout)
            assert output["status"] in {"passed", "wrong_output", "runtime_error", "syntax_error"}
            return output
        except (ValueError, KeyError, AssertionError):
            return {"status": "runner_error", "passed": 0, "total": len(tests),
                    "feedback": f"Malformed runner result: {cp.stdout[-500:]}", "failures": []}


if __name__ == "__main__":
    tests = ["assert find_max([1,5,3]) == 5", "assert find_max([-4, -2]) == -2"]
    wrong = run_visible_tests("def find_max(a):\n    return min(a)\n", tests)
    right = run_visible_tests("def find_max(a):\n    return max(a)\n", tests)
    print("Wrong:", wrong)
    print("Right:", right)
    assert wrong["status"] == "wrong_output"
    assert right["status"] == "passed"


In [ ]:
%%writefile /kaggle/working/agent_loop.py
"""Gemma 4 multi-turn function-repair loop (day-one research prototype)."""
from __future__ import annotations

import re
from typing import Any

from test_runner import run_visible_tests


def extract_python(text: str) -> str:
    """Best effort; keep a record of raw responses for audit."""
    fences = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text, re.S | re.I)
    if fences:
        return fences[-1].strip()
    text = text.strip().replace("<|im_end|>", "").strip()
    # If Gemma adds a preface, discard it before the first code-like line.
    match = re.search(r"(?m)^(?:from\s+\S+\s+import|import\s+\S+|def\s+\w+|class\s+\w+|@\w+)", text)
    return text[match.start():].strip() if match else text


def build_initial_messages(problem: dict) -> list[dict[str, str]]:
    examples = "\n".join(problem.get("test_list", [])[:3])
    return [
        {"role": "system", "content": "Write correct Python code only. Do not include Markdown or explanations. "
            "Include all necessary imports and define the function exactly as requested."},
        {"role": "user", "content": f"Problem:\n{problem['text']}\n\nVisible tests:\n{examples}\n\nWrite Python code that passes these tests."},
    ]


def run_agent(problem: dict, generate_fn, max_attempts: int = 3, max_new_tokens: int = 384) -> dict:
    """generate_fn(messages, max_new_tokens) -> (response_text, input_tokens, output_tokens)."""
    assert 1 <= max_attempts <= 3
    messages = build_initial_messages(problem)
    history = []
    for attempt in range(1, max_attempts + 1):
        raw, prompt_tokens, completion_tokens = generate_fn(messages, max_new_tokens)
        code = extract_python(raw)
        outcome = run_visible_tests(code, problem["test_list"], problem.get("test_setup_code") or "")
        item = {
            "attempt": attempt, "code": code, "raw_output": raw,
            "status": outcome["status"], "feedback": outcome["feedback"],
            "visible_passed": outcome["passed"], "visible_total": outcome["total"],
            "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens,
        }
        history.append(item)
        if outcome["status"] == "passed":
            break
        messages.append({"role": "assistant", "content": code})
        messages.append({"role": "user", "content": "The previous code FAILED the visible tests.\n"
                        + outcome["feedback"] + "\nProvide a corrected full Python solution. "
                        "Do not repeat the same code. Output code only."})
    return {
        "task_id": int(problem["task_id"]), "problem": problem["text"],
        "solved_visible": history[-1]["status"] == "passed", "history": history,
        "attempts_used": len(history),
        "total_completion_tokens": sum(h["completion_tokens"] for h in history),
        "total_prompt_tokens": sum(h["prompt_tokens"] for h in history),
        "note": "Visible-test success only; not MBPP+ or hidden-test success.",
    }


In [ ]:
%%writefile /kaggle/working/day4_eval.py
"""Day 4: paired, held-out MBPP+ recovery evaluation for Gemma 4.

Generate on visible ORIGINAL MBPP tests. Score with official EvalPlus ONLY
in a separate post-generation subprocess. Never include hidden feedback
in model prompts. Use only on disposable, credential-free runtimes.
"""
from __future__ import annotations

import contextlib
import hashlib
import json
import os
import random
import shutil
import sys
import time
from collections import Counter
from pathlib import Path
from zipfile import ZipFile

from agent_loop import build_initial_messages, extract_python
from test_runner import run_visible_tests

BASE_ID = 'google/gemma-4-E4B-it'
MODES = ('c0', 'c1', 'c2')


def write_json(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    temp = p.with_suffix(p.suffix + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')
    temp.replace(p)


def read_jsonl(path):
    p = Path(path)
    if not p.exists():
        return []
    rows = []
    for i, ln in enumerate(p.read_text(encoding='utf-8').splitlines(), 1):
        if ln.strip():
            try:
                rows.append(json.loads(ln))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Corrupt checkpoint {p} at line {i}: {exc}') from exc
    return rows


def append_jsonl(path, row):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    with p.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
        f.flush()
        os.fsync(f.fileno())


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def _copy_selected_from_zip(zip_path, base, condition_dirs=False):
    """Safely extract a small allowlist; never do unrestricted extractall."""
    wanted_files = {'mbpp_split_manifest.json', 'training_plan.json'}
    out = Path(base)
    with ZipFile(zip_path, 'r') as z:
        if z.testzip() is not None:
            raise ValueError(f'Corrupt ZIP: {zip_path}')
        for item in z.infolist():
            if item.is_dir():
                continue
            parts = Path(item.filename).parts
            if '__MACOSX' in parts or '..' in parts:
                continue
            name = parts[-1]
            if name in wanted_files:
                (out / name).write_bytes(z.read(item))
            for cond in ('c1', 'c2'):
                if (f'adapter_{cond}' in parts and
                    name in {'adapter_model.safetensors', 'adapter_config.json'}):
                    dst = out / f'adapter_{cond}'
                    dst.mkdir(exist_ok=True)
                    (dst / name).write_bytes(z.read(item))


def restore_inputs(kaggle_root='/kaggle/input', out='/kaggle/working/day4_inputs'):
    """Locate saved Day 2 and Day 3 outputs or attached backup ZIPs.

    Add Day 2 AND Day 3 saved notebook outputs via Kaggle Add Input, or
    attach private Kaggle Datasets containing gemma_day2_backup.zip and
    gemma_day3_backup.zip.
    """
    root = Path(kaggle_root)
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    required = lambda: all((out / f'adapter_{c}' / 'adapter_model.safetensors').is_file() and
                           (out / f'adapter_{c}' / 'adapter_config.json').is_file()
                           for c in ('c1', 'c2')) and (out/'mbpp_split_manifest.json').is_file()
    if required():
        return out
    if not root.exists():
        raise FileNotFoundError(f'No Kaggle input directory at {root}')
    # Loose files from notebook output: copy adapters and manifest.
    for p in sorted(root.rglob('mbpp_split_manifest.json')):
        if p.is_file():
            shutil.copy2(p, out/'mbpp_split_manifest.json')
            break
    for p in sorted(root.rglob('training_plan.json')):
        if p.is_file():
            shutil.copy2(p, out/'training_plan.json')
            break
    for cond in ('c1', 'c2'):
        configs = sorted(p for p in root.rglob('adapter_config.json') if p.parent.name == f'adapter_{cond}')
        for p in configs:
            w = p.parent/'adapter_model.safetensors'
            if w.is_file():
                dst = out/f'adapter_{cond}'; dst.mkdir(exist_ok=True)
                shutil.copy2(p, dst/p.name)
                shutil.copy2(w, dst/w.name)
                break
    # ZIP backups: only read trusted allowlisted files.
    for pattern in ('gemma_day2_backup*.zip', 'gemma_day3_backup*.zip'):
        for zpath in sorted(root.rglob(pattern)):
            _copy_selected_from_zip(zpath, out)
    if not required():
        missing = [str(p) for p in [out/'mbpp_split_manifest.json',
                  out/'adapter_c1/adapter_config.json',out/'adapter_c1/adapter_model.safetensors',
                  out/'adapter_c2/adapter_config.json',out/'adapter_c2/adapter_model.safetensors'] if not p.exists()]
        raise FileNotFoundError('Attach Day 2 and Day 3 output data. Missing: ' + str(missing))
    a1 = json.loads((out/'adapter_c1/adapter_config.json').read_text())
    a2 = json.loads((out/'adapter_c2/adapter_config.json').read_text())
    assert a1.get('base_model_name_or_path') == BASE_ID and a2.get('base_model_name_or_path') == BASE_ID, 'Adapter base checkpoint mismatch'
    return out


def select_dev_problems(inputs, output, n=30, seed=20261108):
    """Freeze a development set using the original MBPP test split.

    MBPP+ task IDs identify eligible held-out tasks, but NO hidden input/output
    is loaded during generation. Reference code is only used for a local runner
    compatibility check; exclude tasks if that runner cannot execute their
    reference code correctly. Report these exclusions.
    """
    from datasets import load_dataset
    from tqdm.auto import tqdm
    inputs = Path(inputs); output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    manifest = json.loads((inputs/'mbpp_split_manifest.json').read_text())
    dev_path = output/'eval_manifest.json'
    ds = load_dataset('google-research-datasets/mbpp', 'full')
    rows = {int(row['task_id']): dict(row) for row in ds['test']}
    eval_ids = {int(x) for x in manifest['evalplus_mbppplus_ids']}
    trained_ids = set(manifest['mbppplus_decontaminated']['train']['task_ids'])
    candidate_ids = sorted(set(rows).intersection(eval_ids) - trained_ids)
    assert len(candidate_ids) >= n, f'Only {len(candidate_ids)} compatible ID candidates, requested {n}'
    if dev_path.exists():
        frozen = json.loads(dev_path.read_text())
        assert frozen['requested_n'] == n and frozen['seed'] == seed, 'Dev manifest already frozen for different settings; use new output directory'
        chosen = frozen['task_ids']
        assert set(chosen).issubset(candidate_ids), 'Frozen task set no longer matches dataset'
        return [rows[tid] for tid in chosen], frozen
    order = list(candidate_ids)
    random.Random(seed).shuffle(order)
    chosen, excluded = [], []
    for tid in tqdm(order, desc='Check visible-test runner compatibility', unit='task'):
        row = rows[tid]
        oracle = run_visible_tests(row['code'], row['test_list'], row.get('test_setup_code') or '')
        if oracle['status'] == 'passed':
            chosen.append(tid)
            if len(chosen) == n:
                break
        else:
            excluded.append({'task_id': tid, 'reason': oracle['status']})
    if len(chosen) < n:
        raise ValueError(f'Only {len(chosen)} local-runner-compatible tasks. Lower requested n or inspect runner.')
    frozen = {
        'name':'MBPP+ overlap with original MBPP test split, held-out dev set',
        'task_ids': chosen, 'seed': seed, 'requested_n': n,
        'candidate_count': len(candidate_ids),
        'runner_exclusions': excluded,
        'training_overlap': sorted(set(chosen).intersection(trained_ids)),
        'visible_tests': 'original MBPP test_list',
        'hidden_tests': 'official EvalPlus base_input/plus_input, scored only after generation',
        'do_not_use_for_final_claims': 'Use separate untouched test tasks in final evaluation after development is frozen.',
    }
    assert not frozen['training_overlap']
    write_json(dev_path, frozen)
    return [rows[tid] for tid in chosen], frozen


def load_all_adapters(inputs, model_id=BASE_ID):
    import torch
    from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig
    from peft import PeftModel
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    print('Loading Gemma 4 in 4-bit on GPU 0 ...', flush=True)
    base = AutoModelForMultimodalLM.from_pretrained(
        model_id, quantization_config=quant, device_map={'':0}, dtype=torch.float16)
    base.config.use_cache = True
    processor = AutoProcessor.from_pretrained(model_id)
    model = PeftModel.from_pretrained(base, str(Path(inputs)/'adapter_c1'),
        adapter_name='c1', is_trainable=False)
    model.load_adapter(str(Path(inputs)/'adapter_c2'), adapter_name='c2', is_trainable=False)
    model.eval()
    model.set_adapter('c1')
    assert {'c1','c2'} <= set(model.peft_config.keys()), f'Adapters not loaded: {model.peft_config.keys()}'
    print('Loaded C1 + C2. GPU0 allocated:', round(torch.cuda.memory_allocated(0)/1024**3, 2), 'GiB', flush=True)
    return model, processor


def generate_code(model, processor, messages, condition, max_new_tokens=384):
    """Deterministic inference; C0 disables all adapters temporarily."""
    import torch
    if condition not in MODES:
        raise ValueError(condition)
    if condition in ('c1','c2'):
        model.set_adapter(condition)
        ctx = contextlib.nullcontext()
    else:
        ctx = model.disable_adapter()
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True, return_tensors='pt',
        add_generation_prompt=True, enable_thinking=False).to('cuda:0')
    n = int(inputs['input_ids'].shape[1])
    started = time.perf_counter()
    with ctx, torch.inference_mode():
        out = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens,
            pad_token_id=processor.tokenizer.pad_token_id)
    new = out[0][n:]
    return {
        'raw': processor.decode(new, skip_special_tokens=True),
        'prompt_tokens':n, 'completion_tokens':int(len(new)),
        'generation_seconds':round(time.perf_counter()-started, 3),
    }


def record_base_initials(problems, model, processor, output, max_new_tokens=384):
    from tqdm.auto import tqdm
    output = Path(output); output.mkdir(parents=True, exist_ok=True)
    log = output/'base_initials.jsonl'
    existing = {int(r['task_id']):r for r in read_jsonl(log)}
    if len(existing)!=len(read_jsonl(log)):
        raise ValueError('Duplicate base initial task IDs found in checkpoint')
    pending = [p for p in problems if int(p['task_id']) not in existing]
    with tqdm(total=len(problems), initial=len(problems)-len(pending),
              desc='C0 first attempts', unit='problem', dynamic_ncols=True) as bar:
        for p in pending:
            tid = int(p['task_id'])
            bar.set_postfix(task=tid)
            print(f'Generating initial code for MBPP {tid}', flush=True)
            g = generate_code(model, processor, build_initial_messages(p), 'c0', max_new_tokens)
            code = extract_python(g['raw'])
            check = run_visible_tests(code, p['test_list'], p.get('test_setup_code') or '')
            rec = {'task_id':tid,'condition':'c0','attempt':1,'code':code,
                   'visible_status':check['status'], 'visible_feedback':check['feedback'],
                   'visible_passed':check.get('passed',0), 'visible_total':check.get('total',0),
                   **g}
            append_jsonl(log, rec)
            existing[tid]=rec
            bar.update()
            bar.set_postfix(task=tid, visible=check['status'], sec=g['generation_seconds'])
    return existing


def _feedback_message(feedback):
    return 'The previous code FAILED the visible tests.\n' + feedback + '\nProvide a corrected full Python solution. Do not repeat the same code. Output code only.'


def record_fixed_corrections(problems, initials, model, processor, output, max_new_tokens=384, max_corrections=1):
    """Same failed C0 code and same initial feedback for C0/C1/C2.

    No corrections are issued after the visible tests pass, regardless of
    eventual hidden-test success/failure. Hidden results are never in prompts.
    """
    from tqdm.auto import tqdm
    output = Path(output); output.mkdir(parents=True, exist_ok=True)
    log=output/'fixed_corrections.jsonl'
    existing_rows=read_jsonl(log)
    existing={(int(r['task_id']),r['condition']):r for r in existing_rows}
    if len(existing)!=len(existing_rows):
        raise ValueError('Duplicate fixed-state records in checkpoint')
    tasks=[p for p in problems if initials[int(p['task_id'])]['visible_status']!='passed']
    count=len(tasks)*len(MODES)
    assert tasks, 'No visible failures in chosen tasks. Increase number of evaluation tasks; do not change their IDs after looking at hidden results.'
    pending=sum((int(p['task_id']),c) not in existing for p in tasks for c in MODES)
    with tqdm(total=count,initial=count-pending,desc='Paired recovery conditions',unit='condition',dynamic_ncols=True) as bar:
        for p in tasks:
            tid=int(p['task_id'])
            init=initials[tid]
            if init['visible_status']=='runner_error':
                raise ValueError(f'Runner error on task {tid}; inspect before evaluation')
            for condition in MODES:
                key=(tid,condition)
                if key in existing:
                    continue
                messages=build_initial_messages(p)
                messages.append({'role':'assistant','content':init['code']})
                messages.append({'role':'user','content':_feedback_message(init['visible_feedback'])})
                attempts=[]
                for retry in range(1,max_corrections+1):
                    print(f'Fixing MBPP {tid}: {condition}, correction {retry}/{max_corrections}',flush=True)
                    g=generate_code(model,processor,messages,condition,max_new_tokens)
                    code=extract_python(g['raw'])
                    chk=run_visible_tests(code,p['test_list'],p.get('test_setup_code') or '')
                    attempts.append({'correction_index':retry,'code':code,
                                     'visible_status':chk['status'], 'visible_feedback':chk['feedback'],
                                     'visible_passed':chk.get('passed',0),
                                     'visible_total':chk.get('total',0), **g})
                    if chk['status']=='passed':
                        break
                    if retry < max_corrections:
                        messages.extend([
                            {'role':'assistant','content':code},
                            {'role':'user','content':_feedback_message(chk['feedback'])}])
                rec={'task_id':tid,'condition':condition,
                     'starting_code':init['code'],
                     'starting_feedback':init['visible_feedback'],
                     'starting_visible_status':init['visible_status'],
                     'attempts':attempts,
                     'recovered_visible':any(a['visible_status']=='passed' for a in attempts),
                     'num_corrections':len(attempts),
                     'total_completion_tokens':sum(a['completion_tokens'] for a in attempts),
                     'total_prompt_tokens':sum(a['prompt_tokens'] for a in attempts)}
                append_jsonl(log,rec);existing[key]=rec
                bar.update()
                bar.set_postfix(task=tid,condition=condition,
                                visible=rec['recovered_visible'])
    return existing


def prepare_offline_scoring(problems, output):
    """Create evaluation inputs after all completions were generated."""
    output=Path(output)
    ids={int(p['task_id']) for p in problems}
    initials=read_jsonl(output/'base_initials.jsonl')
    fixed=read_jsonl(output/'fixed_corrections.jsonl')
    assert {int(r['task_id']) for r in initials} == ids, 'Initial generation incomplete'
    failed={int(r['task_id']) for r in initials if r['visible_status']!='passed'}
    for cond in MODES:
        assert {int(r['task_id']) for r in fixed if r['condition']==cond} == failed, f'Missing fixed corrections for {cond}'
    rows=[]
    for r in initials:
        rows.append({'score_key':f'initial:{int(r["task_id"])}',
                     'task_id':int(r['task_id']), 'condition':'c0',
                     'stage':'initial', 'attempt':1,'code':r['code']})
    for r in fixed:
        for a in r['attempts']:
            rows.append({'score_key':f'fixed:{r["condition"]}:{int(r["task_id"])}:{a["correction_index"]}',
                         'task_id':int(r['task_id']), 'condition':r['condition'],
                         'stage':'fixed_recovery','attempt':a['correction_index'],
                         'code':a['code']})
    assert len(rows)==len({r['score_key'] for r in rows}), 'Duplicate score keys'
    p=output/'scoring_inputs.jsonl'
    tmp=p.with_suffix('.jsonl.tmp')
    tmp.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows),encoding='utf-8')
    tmp.replace(p)
    return p


def _rate(arr):
    return sum(arr)/len(arr) if arr else float('nan')


def paired_bootstrap(c1,c2,n_boot=3000,seed=42):
    assert len(c1)==len(c2)>0
    diffs=[int(b)-int(a) for a,b in zip(c1,c2)]
    rng=random.Random(seed)
    draws=[]
    for _ in range(n_boot):
        draws.append(sum(diffs[rng.randrange(len(diffs))] for _ in diffs)/len(diffs))
    draws.sort()
    return {'difference_c2_minus_c1':sum(diffs)/len(diffs),
            'ci_low':draws[int(0.025*n_boot)],
            'ci_high':draws[min(n_boot-1,int(0.975*n_boot))],
            'n_pairs':len(diffs),
            'discordant_c2_only':sum(d==1 for d in diffs),
            'discordant_c1_only':sum(d==-1 for d in diffs)}


def analyze_official(problems, output):
    output=Path(output)
    initials=read_jsonl(output/'base_initials.jsonl')
    corrections=read_jsonl(output/'fixed_corrections.jsonl')
    official=read_jsonl(output/'official_scores.jsonl')
    im={r['score_key']:r for r in official}
    expected_keys={r['score_key'] for r in read_jsonl(output/'scoring_inputs.jsonl')}
    if set(im)!=expected_keys or len(im)!=len(official):
        raise ValueError('Incomplete/duplicate official score log. Rerun scorer.')
    failed_ids=sorted(int(r['task_id']) for r in initials if r['visible_status']!='passed')
    f={(int(r['task_id']),r['condition']):r for r in corrections}
    successes={}
    for cond in MODES:
        hits=[];visible=[];tokens=[]
        for tid in failed_ids:
            r=f[(tid,cond)]
            hits.append(any(im[f'fixed:{cond}:{tid}:{a["correction_index"]}']['mbppplus_pass'] for a in r['attempts']))
            visible.append(r['recovered_visible'])
            tokens.append(r['total_completion_tokens'])
        successes[cond]={'official_recovery':sum(hits), 'n_failed_states':len(failed_ids),
             'recovery_rate':_rate(hits), 'visible_recovery_rate':_rate(visible),
             'mean_completion_tokens':_rate(tokens)}
    c1=[];c2=[]
    for tid in failed_ids:
        for arr,cond in ((c1,'c1'),(c2,'c2')):
            r=f[(tid,cond)]
            arr.append(any(im[f'fixed:{cond}:{tid}:{a["correction_index"]}']['mbppplus_pass'] for a in r['attempts']))
    base_pass=sum(im[f'initial:{int(r["task_id"])}']['mbppplus_pass'] for r in initials)
    summary={
        'benchmark': 'EvalPlus MBPP+ official base + plus oracle',
        'dev_only':True, 'n_dev_tasks':len(problems),
        'n_fixed_failed_states':len(failed_ids),
        'base_first_attempt_official_pass':base_pass,
        'base_first_attempt_official_rate':base_pass/len(initials),
        'fixed_state_results':successes,
        'paired_C2_vs_C1':paired_bootstrap(c1,c2),
        'methodology_note':'Fixed starting states are first-attempt visible failures from C0; hidden tests are only scored after all generations and never returned as feedback.',
    }
    write_json(output/'day4_summary.json',summary)
    return summary


In [ ]:
%%writefile /kaggle/working/day4_score.py
"""Offline official EvalPlus MBPP+ scoring for Day 4.

Run ONLY after generation, in a separate CPU subprocess. No hidden feedback
is supplied to the coding agent. IMPORTANT: EvalPlus executes generated code;
run only in an isolated, credential-free container/runtime.
"""
from __future__ import annotations
import argparse, hashlib, json, sys
from pathlib import Path
from tqdm.auto import tqdm

from day4_eval import append_jsonl, read_jsonl, sha256_bytes


def score(input_jsonl, output_jsonl):
    # Import lazily: model-generation process does not load hidden oracle.
    from evalplus.data import get_mbpp_plus, get_mbpp_plus_hash
    from evalplus.evaluate import check_correctness, get_groundtruth
    from evalplus.eval._special_oracle import MBPP_OUTPUT_NOT_NONE_TASKS
    source=Path(input_jsonl)
    dest=Path(output_jsonl)
    requests=read_jsonl(source)
    if not requests:
        raise RuntimeError(f'No scoring requests: {source}')
    score_keys=[r['score_key'] for r in requests]
    if len(set(score_keys))!=len(score_keys):
        raise ValueError('Duplicate score keys')
    cache_inputs=dest.with_suffix('.source_sha256')
    hash_input=sha256_bytes(source.read_bytes())
    if cache_inputs.exists() and cache_inputs.read_text().strip()!=hash_input:
        raise RuntimeError('Score requests changed since scoring began; do not mix checkpoints')
    cache_inputs.parent.mkdir(parents=True,exist_ok=True)
    cache_inputs.write_text(hash_input+'\n')
    old=read_jsonl(dest)
    if len(old)!=len({r['score_key'] for r in old}):
        raise ValueError('Duplicate score checkpoints')
    done={r['score_key']:r for r in old}
    if not set(done).issubset(score_keys):
        raise ValueError('Checkpoint contains scores for different task set')
    problems_all=get_mbpp_plus()
    key_by_id = {int(str(key).split('/')[-1]): key for key in problems_all}
    requested_ids = {int(r['task_id']) for r in requests}
    assert requested_ids.issubset(key_by_id), 'Some selected IDs are absent from official EvalPlus MBPP+'
    ids = {key_by_id[tid] for tid in requested_ids}
    problems={tid:problems_all[tid] for tid in ids}
    print('Loaded',len(problems),'official EvalPlus MBPP+ problems',flush=True)
    # Hash includes dataset version AND selected subset: NEVER corrupt full-benchmark cache.
    subhash=hashlib.sha256((get_mbpp_plus_hash() + '|' + ','.join(sorted(ids))).encode()).hexdigest()[:24]
    expected=get_groundtruth(problems,'day4_dev_subset_'+subhash,MBPP_OUTPUT_NOT_NONE_TASKS)
    remaining=[r for r in requests if r['score_key'] not in done]
    with tqdm(total=len(requests),initial=len(done),desc='Official EvalPlus scoring',unit='solution',dynamic_ncols=True) as bar:
        for r in remaining:
            tid=key_by_id[int(r['task_id'])]
            try:
                result=check_correctness(
                    'mbpp', 0, problems[tid], r['code'], expected[tid],
                    base_only=False, fast_check=True,
                )
                base_status=result['base'][0]
                plus_status=result['plus'][0]
            except Exception as exc:
                # Treat scorer exceptions as infrastructure issues, not incorrect code.
                raise RuntimeError(f'Official evaluator failed on {r["score_key"]}: {exc}') from exc
            rec={'score_key':r['score_key'],'task_id':int(r['task_id']),
                 'stage':r['stage'],'condition':r['condition'],
                 'evalplus_base_status':base_status,
                 'evalplus_plus_status':plus_status,
                 'mbppplus_pass':base_status=='pass' and plus_status=='pass'}
            append_jsonl(dest,rec)
            bar.update()
            bar.set_postfix(task=tid,pass_plus=rec['mbppplus_pass'])
    print('Finished:',len(read_jsonl(dest)),'official scores saved at',dest,flush=True)


if __name__=='__main__':
    parser=argparse.ArgumentParser()
    parser.add_argument('--requests',required=True)
    parser.add_argument('--output',required=True)
    args=parser.parse_args()
    score(args.requests,args.output)


## D — Restore adapters and the **frozen** Day 5 split (required)

In [ ]:
import sys, json, shutil
from pathlib import Path
from zipfile import ZipFile
sys.path.insert(0,'/kaggle/working')
from day4_eval import restore_inputs,write_json,read_jsonl
INPUTS=restore_inputs('/kaggle/input', INPUTS)
root=Path('/kaggle/input')
# Prefer a saved Day 5 notebook output; only accept a day5_outputs path.
manifest_candidates=[p for p in root.rglob('day5_partition_manifest.json')
                     if p.is_file() and 'day5_outputs' in p.parts]
manifests=[]
for p in manifest_candidates:
    manifests.append(json.loads(p.read_text()))
# Also allow the ZIP backup attached as a Kaggle dataset.
for p in root.rglob('gemma_day5_backup*.zip'):
    with ZipFile(p) as z:
        name='gemma_day5_backup/day5_outputs/day5_partition_manifest.json'
        if name in z.namelist():
            manifests.append(json.loads(z.read(name)))
if not manifests:
    raise FileNotFoundError('Attach saved Day 5 outputs or gemma_day5_backup.zip; no frozen partition manifest found.')
assert all(x['final_holdout_ids']==manifests[0]['final_holdout_ids'] for x in manifests), 'Conflicting Day 5 final holdout manifests'
partition=manifests[0]
ids=[int(x) for x in partition['final_holdout_ids']]
assert len(ids)==70 and len(set(ids))==70
assert not set(ids)&(set(partition['new_dev_ids'])|set(partition['prior_day4_dev_ids']))
assert not partition.get('training_overlap')
write_json(OUT/'frozen_manifest.json',partition)
print('Frozen final holdout IDs:',len(ids))
print('Example IDs:',ids[:8])
print('C1 adapter:',(INPUTS/'adapter_c1/adapter_model.safetensors').stat().st_size,'bytes')
print('C2 adapter:',(INPUTS/'adapter_c2/adapter_model.safetensors').stat().st_size,'bytes')


## E — Load only original MBPP problem statements and visible tests

In [ ]:
from datasets import load_dataset
from test_runner import run_visible_tests

mbpp = load_dataset('google-research-datasets/mbpp', 'full')
public = {int(r['task_id']): dict(r) for r in mbpp['test']}
train_manifest = json.loads((INPUTS / 'mbpp_split_manifest.json').read_text())
allowed = set(map(int, train_manifest['evalplus_mbppplus_ids']))
trained = set(map(int, train_manifest['mbppplus_decontaminated']['train']['task_ids']))

# These protections must remain in place: holdout IDs are fixed and disjoint.
assert set(ids).issubset(set(public) & allowed)
assert not set(ids) & trained
problems = [public[i] for i in ids]
assert len(problems) == 70 and len({int(p['task_id']) for p in problems}) == 70

# Diagnostic only. Do not exclude frozen tasks based on reference visible-test status:
# a local runner's strict equality or environment can disagree with the benchmark.
check = []
for p in problems:
    result = run_visible_tests(p['code'], p['test_list'], p.get('test_setup_code') or '')
    check.append({
        'task_id': int(p['task_id']),
        'reference_visible_status': result['status'],
        'reference_visible_feedback': result.get('feedback', '')[:1000],
    })
write_json(OUT / 'runner_compatibility.json', check)

mismatches = [item for item in check if item['reference_visible_status'] != 'passed']
print('Frozen holdout tasks:', len(problems))
print('References passing local visible tests:', len(check) - len(mismatches), '/', len(check))
print('Reference-visible mismatches:', len(mismatches))
if mismatches:
    print('\nReference-visible mismatch diagnostics:')
    for item in mismatches[:20]:
        print(f"  task={item['task_id']} status={item['reference_visible_status']}")
        print('  feedback:', item['reference_visible_feedback'][:350].replace('\n', ' | '))
    if len(mismatches) > 20:
        print('  Additional mismatches are saved in runner_compatibility.json')
    print('\nWARNING: Local visible tests disagree with some reference solutions.')
    print('Keeping ALL 70 frozen task IDs; no post-hoc exclusions.')
    print('Use official EvalPlus for final scoring and disclose feedback inconsistencies.')
else:
    print('All frozen reference solutions passed local visible tests.')

# If the failures are runner-level errors affecting most tasks, diagnose the runner
# before running expensive inference, rather than assuming these are harmless.
runner_failures = [x for x in check if x['reference_visible_status'] == 'runner_error']
if runner_failures:
    raise RuntimeError(f'{len(runner_failures)} runner_error records: inspect runner_compatibility.json before proceeding')


## F — Load C0, C1 and C2 on one 4-bit Gemma

In [ ]:
from day4_eval import load_all_adapters, BASE_ID
model, processor=load_all_adapters(INPUTS,BASE_ID)
print('Loaded:',list(model.peft_config))


## G — Generate C0 first attempts (checkpoint after every task)

In [ ]:
from day4_eval import record_base_initials
initials=record_base_initials(problems,model,processor,OUT,max_new_tokens=MAX_NEW_TOKENS)
assert len(initials)==70
print('Initially failed visible tests:',sum(x['visible_status']!='passed' for x in initials.values()))


## H — Generate 1 paired correction per visible-failing C0 state

In [ ]:
from day4_eval import record_fixed_corrections
fixed=record_fixed_corrections(problems,initials,model,processor,OUT,max_new_tokens=MAX_NEW_TOKENS,max_corrections=1)
fail_count=sum(x['visible_status']!='passed' for x in initials.values())
assert len(fixed)==fail_count*3, (len(fixed),fail_count*3)
print('Saved paired corrections:',len(fixed))


## I — Immediate generation checkpoint backup (download this now)

In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
from IPython.display import FileLink,display
checkpoint=Path('/kaggle/working/final_holdout_generation_checkpoint.zip')
with ZipFile(checkpoint,'w',ZIP_DEFLATED) as z:
    for name in ['frozen_manifest.json','runner_compatibility.json','base_initials.jsonl','fixed_corrections.jsonl']:
        path=OUT/name
        if not path.is_file():raise FileNotFoundError(path)
        z.write(path,arcname='final_holdout_outputs/'+name)
with ZipFile(checkpoint) as z:assert z.testzip() is None
print('Saved:',checkpoint);display(FileLink(str(checkpoint)))


## J — Prepare official scoring requests **after all generation is complete**

In [ ]:
from day4_eval import prepare_offline_scoring
scoring_inputs=prepare_offline_scoring(problems,OUT)
print('Requests:',len(read_jsonl(scoring_inputs)))


## K — Official EvalPlus base+plus scoring (separate CPU-only process)

In [ ]:
import subprocess,os,sys
score_file=OUT/'official_scores.jsonl'
child_env=dict(os.environ,CUDA_VISIBLE_DEVICES='')
subprocess.run([sys.executable,'/kaggle/working/day4_score.py',
                '--requests',str(scoring_inputs),'--output',str(score_file)],
               check=True, env=child_env)
assert len(read_jsonl(score_file))==len(read_jsonl(scoring_inputs))
print('Official scores complete:',score_file)


## L — Audited holdout summary (no training or model choice)

In [ ]:
from day4_eval import paired_bootstrap
score_rows=read_jsonl(OUT/'official_scores.jsonl')
sc={r['score_key']:bool(r['mbppplus_pass']) for r in score_rows}
assert len(sc)==len(score_rows)
failed=sorted(t for t,r in initials.items() if r['visible_status']!='passed')
false_alarms=sorted(t for t in failed if sc[f'initial:{t}'])
genuinely_incorrect=[t for t in failed if not sc[f'initial:{t}']]
print('Visible failed starts:',len(failed),'officially correct among these:',len(false_alarms))
print('Audited incorrect starting states:',len(genuinely_incorrect))
by_model={}
for c in ('c0','c1','c2'):
    vals=[sc[f'fixed:{c}:{t}:1'] for t in genuinely_incorrect]
    by_model[c]={'passed':sum(vals),'n':len(vals),'rate':sum(vals)/len(vals) if vals else None}
ci=paired_bootstrap([sc[f'fixed:c1:{t}:1'] for t in genuinely_incorrect],
                    [sc[f'fixed:c2:{t}:1'] for t in genuinely_incorrect],n_boot=20000,seed=20261110) if genuinely_incorrect else None
report={
 'scope':'FROZEN FINAL HOLDOUT -- selected before Day 5 evaluation; no tuning performed',
 'n_tasks':len(problems),
 'first_attempt_official_pass':sum(sc[f'initial:{t}'] for t in ids),
 'visible_failed_starts':len(failed),
 'officially_correct_visible_fail_ids':false_alarms,
 'true_failed_starts':len(genuinely_incorrect),
 'by_model':by_model,'paired_C2_vs_C1':ci,
 'official_scoring_requests':len(score_rows),
 'note':'Hidden tests only evaluated after frozen generations. Distinguish protocol-defined visible failure states from post-hoc true incorrect states.'}
write_json(OUT/'final_holdout_summary.json',report)
print(json.dumps(report,indent=2))


## M — Final backup; **Save Version → Quick Save with outputs**

In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
from IPython.display import FileLink,display
zip_path=Path('/kaggle/working/gemma_final_holdout_backup.zip')
files=sorted(p for p in OUT.rglob('*') if p.is_file())
with ZipFile(zip_path,'w',ZIP_DEFLATED) as z:
    for f in files:z.write(f,arcname=str(f.relative_to('/kaggle/working')))
with ZipFile(zip_path) as z:
    assert z.testzip() is None
print('Backed up',len(files),'output files; size',round(zip_path.stat().st_size/1024**2,2),'MiB')
display(FileLink(str(zip_path)))
